In [ ]:
import pandas as pd
import json
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

tqdm.pandas()

# ============================================================
# 0) UTILITIES
# ============================================================

def extract_text(tweet):
    txt = tweet.get("text", "")
    if not isinstance(txt, str):
        txt = ""
    return txt


def predict_batch(text_list, tokenizer, model, device="cuda", batch_size=32):
    model.eval()
    preds = []
    probs_list = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]

        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            out = model(**enc)
            probs = torch.softmax(out.logits, dim=1).cpu().numpy()
            pred  = np.argmax(probs, axis=1)

        preds.extend(pred)
        probs_list.extend(probs)

    return np.array(preds), np.array(probs_list)


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)



# ============================================================
# 1) LOAD TRAIN
# ============================================================

train_path = "/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/train.jsonl"
df = load_jsonl(train_path)

df["challenge_id"] = df["challenge_id"].astype(str)

df["full_txt"] = df.progress_apply(extract_text, axis=1)

texts = df["full_txt"].tolist()
labels = df["label"].tolist()

print("Train size =", len(df))



# ============================================================
# 2) SPLIT 75% TRAIN BERT / 25% OOS
# ============================================================

idx_all = df.index.values

idx_train_bert, idx_oos = train_test_split(
    idx_all,
    test_size=0.25,
    stratify=df["label"],
    random_state=42
)

df_train_bert = df.loc[idx_train_bert].reset_index(drop=True)
df_oos        = df.loc[idx_oos].reset_index(drop=True)

print("\nSplit:")
print(" -  BERT train:", len(df_train_bert))
print(" -  OOS stack :", len(df_oos))



# ============================================================
# 3) TOKENIZATION
# ============================================================

tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

train_enc = tokenizer(
    df_train_bert["full_txt"].tolist(),
    truncation=True,
    padding=False,
    max_length=512
)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels": self.labels[idx],
        }


train_dataset = TextDataset(train_enc, df_train_bert["label"].tolist())



# ============================================================
# 4) TRAIN BERT ON 75%
# ============================================================

model = CamembertForSequenceClassification.from_pretrained(
    "camembert-base", num_labels=2
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

training_args = TrainingArguments(
    output_dir="./bert-text-75pct",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=200,
    save_strategy="epoch",
    report_to="none"
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("\n🚀 Training BERT (75% train)...\n")
trainer.train()

trainer.save_model("./bert-text-75pct")
tokenizer.save_pretrained("./bert-text-75pct")



# ============================================================
# 5) OOS PREDICTIONS (STACKING)
# ============================================================

print("\n🔵 Generating OOS predictions...")

oos_texts = df_oos["full_txt"].tolist()

oos_preds, oos_probs = predict_batch(
    oos_texts, tokenizer, model, device=device
)

df_oos["bert_pred"] = oos_preds
df_oos["prob_0"] = oos_probs[:,0]
df_oos["prob_1"] = oos_probs[:,1]
df_oos["bert_conf"] = oos_probs.max(axis=1)



# ============================================================
# 6) PREDICT TEST.JSONL
# ============================================================

print("\n🟢 Predicting TEST...")

test_path = "/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/kaggle_test.jsonl"
df_test_raw = load_jsonl(test_path)
df_test_raw["challenge_id"] = df_test_raw["challenge_id"].astype(str)
df_test_raw["full_txt"] = df_test_raw.progress_apply(extract_text, axis=1)

test_texts = df_test_raw["full_txt"].tolist()

test_preds, test_probs = predict_batch(
    test_texts, tokenizer, model, device=device
)

df_test_raw["bert_pred"] = test_preds
df_test_raw["prob_0"] = test_probs[:,0]
df_test_raw["prob_1"] = test_probs[:,1]
df_test_raw["bert_conf"] = test_probs.max(axis=1)



# ============================================================
# 7) SAVE OUTPUT FILES
# ============================================================

df_oos.to_csv("/kaggle/working/bert_train_oof.csv", index=False)
df_test_raw.to_csv("/kaggle/working/bert_test_predictions.csv", index=False)

print("\n🎉 Saved:")
print(" - bert_train_oof.csv (OOS stacking features)")
print(" - bert_test_predictions.csv (test features)")


In [ ]:
# =============================================================
# STACKING BERT DESCRIPTION (75% TRAIN / 25% OOS / TEST)
# =============================================================

import json
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

tqdm.pandas()

device = "cuda" if torch.cuda.is_available() else "cpu"


# =============================================================
# 1) UTILITIES
# =============================================================

def extract_description(user_field):
    if isinstance(user_field, dict):
        d = user_field.get("description", "")
        return d if isinstance(d, str) else ""
    return ""


def predict_batch(texts, tokenizer, model, batch_size=32):
    model.eval()
    preds = []
    probs_all = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            out = model(**enc)
            prob = torch.softmax(out.logits, dim=1).cpu().numpy()
            pred = np.argmax(prob, axis=1)

        preds.extend(pred.tolist())
        probs_all.extend(prob.tolist())

    return np.array(preds), np.array(probs_all)


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


# =============================================================
# 2) LOAD TRAIN.JSONL + EXTRACT USER DESCRIPTIONS
# =============================================================

train_path = "/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/train.jsonl"
df = load_jsonl(train_path)

df["challenge_id"] = df["challenge_id"].astype(str)
df["description"] = df["user"].progress_apply(extract_description)

# user-level (unique descriptions)
df_users = df.groupby("challenge_id").first().reset_index()
df_users = df_users[["challenge_id", "description", "label"]]

print("Users detected:", len(df_users))


# =============================================================
# 3) SPLIT 75% TRAIN BERT / 25% OOS
# =============================================================

idx_all = df_users.index.values

idx_train, idx_oos = train_test_split(
    idx_all,
    test_size=0.25,
    random_state=42,
    stratify=df_users["label"]
)

df_train_bert = df_users.loc[idx_train].reset_index(drop=True)
df_oos = df_users.loc[idx_oos].reset_index(drop=True)

print("Train BERT :", len(df_train_bert))
print("OOS stack  :", len(df_oos))


# =============================================================
# 4) TOKENIZATION + DATASET
# =============================================================

tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

train_enc = tokenizer(
    df_train_bert["description"].tolist(),
    truncation=True,
    padding=False,
    max_length=512
)

class DescDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels": self.labels[idx],
        }

train_dataset = DescDataset(train_enc, df_train_bert["label"].tolist())


# =============================================================
# 5) TRAIN BERT ON 75%
# =============================================================

model = CamembertForSequenceClassification.from_pretrained(
    "camembert-base", num_labels=2
).to(device)

training_args = TrainingArguments(
    output_dir="./bert-desc-75pct",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=200,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
)

print("\n🚀 Training BERT DESCRIPTION (75% train)...\n")
trainer.train()

trainer.save_model("./bert-desc-75pct")
tokenizer.save_pretrained("./bert-desc-75pct")


# =============================================================
# 6) OOS PREDICTIONS (25%)
# =============================================================

print("\n🔵 Generating DESCRIPTION OOS predictions...")

oos_desc = df_oos["description"].tolist()
oos_pred, oos_probs = predict_batch(oos_desc, tokenizer, model)

df_oos["pred"] = oos_pred
df_oos["prob_0"] = oos_probs[:, 0]
df_oos["prob_1"] = oos_probs[:, 1]
df_oos["confidence"] = oos_probs.max(axis=1)

df_oos.to_csv("/kaggle/working/bert_userdesc_train_oof.csv", index=False)

print("✓ Saved:", "/kaggle/working/bert_userdesc_train_oof.csv")


# =============================================================
# 7) PREDICT TEST.JSONL
# =============================================================

print("\n🟢 Predicting TEST descriptions...")

test_path = "/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/kaggle_test.jsonl"
df_test = load_jsonl(test_path)
df_test["challenge_id"] = df_test["challenge_id"].astype(str)
df_test["description"] = df_test["user"].progress_apply(extract_description)

df_test_users = df_test.groupby("challenge_id").first().reset_index()

test_desc = df_test_users["description"].tolist()

test_pred, test_probs = predict_batch(test_desc, tokenizer, model)

df_test_users["pred"] = test_pred
df_test_users["prob_0"] = test_probs[:, 0]
df_test_users["prob_1"] = test_probs[:, 1]
df_test_users["confidence"] = test_probs.max(axis=1)

df_test_users.to_csv("/kaggle/working/bert_userdesc_test_predictions.csv", index=False)

print("✓ Saved:", "/kaggle/working/bert_userdesc_test_predictions.csv")


print("\n🎉 FINISHED — DESCRIPTION STACKING COMPLETE.")


In [ ]:
# =============================================================
# 8) ACCURACY ON OUT-OF-SAMPLE (25% never seen by BERT)
# =============================================================
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

print("\n===================== OOS PERFORMANCE =====================")

y_true = df_oos["label"].astype(int).values
y_pred = df_oos["pred"].astype(int).values

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary"
)

print(f"🔥 Accuracy OOS (25%): {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1:.4f}")

print("\nClassification report:\n")
print(classification_report(y_true, y_pred))

print("Distribution des prédictions :")
print(df_oos["pred"].value_counts(normalize=True))

print("\nMoyenne des probas :")
print(df_oos[["prob_0", "prob_1"]].mean())
